# 02 — Traits de côte annuels de Saint-Louis (1984-2026)

Ce notebook transforme les 616 traits de côte extraits par CoastSat (notebook 01) en **un trait de côte par année**. Chaque position annuelle est la médiane de toutes les détections de l'année : elle gomme l'effet de la marée, des vagues, de l'écume et des petites erreurs de détection.

**Méthode :**
1. Des transects perpendiculaires au trait de référence, tous les 50 m, de la terre vers la mer.
2. Pour chaque date, l'intersection du trait de côte avec chaque transect.
3. Pour chaque transect et chaque année, une position médiane robuste (valeurs aberrantes écartées).
4. La reconstruction d'un trait de côte par année à partir de ces positions.

**Sorties** (dans `Mon Drive/erosion-cotiere-saint-louis-senegal/`) :
- `data/processed/traits_de_cote_annuels_1984_2026.geojson` : un trait par année (EPSG:32628)
- `data/processed/traits_de_cote_annuels_dsas.geojson` : les années fiables, avec date et incertitude au format DSAS
- `data/processed/ligne_de_base_dsas.geojson` : la ligne de base pour DSAS (côté terre)
- `data/processed/transects.geojson` et `data/processed/positions_annuelles.csv` : positions par transect et par année
- `app/data/traits_de_cote_annuels.geojson` : version allégée en WGS 84 pour l'application web
- `outputs/figures/` : carte des traits annuels et graphique d'évolution

Ce notebook n'utilise pas Earth Engine : il tourne en une ou deux minutes.

## 1. Google Drive et chargement des données

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

import os
import warnings
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
from shapely.geometry import LineString, MultiLineString
from shapely.strtree import STRtree

ROOT = "/content/drive/MyDrive/erosion-cotiere-saint-louis-senegal"
EPSG = 32628

traits = gpd.read_file(os.path.join(ROOT, "data", "processed", "traits_de_cote_saint_louis_1984_2026.geojson")).to_crs(EPSG)
traits = traits[traits.geometry.notna() & ~traits.geometry.is_empty].reset_index(drop=True)
reference = gpd.read_file(os.path.join(ROOT, "data", "aoi", "trait_de_cote_reference.geojson")).to_crs(EPSG)
reference = reference.geometry.iloc[0]
reference = MultiLineString(list(getattr(reference, "geoms", [reference])))

print(f"{len(traits)} traits de côte de {traits.annee.min()} à {traits.annee.max()}")
print(traits.groupby("satname").size().rename("traits").to_string())

## 2. Transects

Un transect tous les 50 m le long du trait de référence (tracé sur 2024), perpendiculaire à la côte. Chaque transect part de 600 m côté terre et va jusqu'à 700 m côté mer, ce qui couvre la tolérance de 500 m utilisée pour les images anciennes.

La direction de chaque transect est calculée sur le trait lissé sur 400 m, pour éviter que des transects voisins se croisent dans les virages (pointe de la Langue de Barbarie).

In [ ]:
ESPACEMENT = 50     # m entre deux transects
COTE_TERRE = 600    # m côté terre depuis le trait de référence
COTE_MER = 700      # m côté mer
LISSAGE = 400       # m, fenêtre de calcul de la direction

def creer_transects(ref, espacement=ESPACEMENT, cote_terre=COTE_TERRE, cote_mer=COTE_MER, lissage=LISSAGE):
    transects = []
    for t, ligne in enumerate(ref.geoms):
        L = ligne.length
        if L < espacement:
            continue
        for d in np.arange(espacement / 2, L, espacement):
            p = np.array(ligne.interpolate(d).coords[0][:2])
            a = np.array(ligne.interpolate(max(d - lissage / 2, 0)).coords[0][:2])
            b = np.array(ligne.interpolate(min(d + lissage / 2, L)).coords[0][:2])
            tangente = (b - a) / np.linalg.norm(b - a)
            normale = np.array([-tangente[1], tangente[0]])
            if normale[0] > 0:          # normale orientée vers la mer, à l'ouest
                normale = -normale
            transects.append({"troncon": t, "distance_le_long": round(float(d)),
                              "geometry": LineString([p - normale * cote_terre, p + normale * cote_mer])})
    gdf = gpd.GeoDataFrame(transects, crs=EPSG)
    gdf.insert(0, "id_transect", np.arange(len(gdf)))
    return gdf

transects = creer_transects(reference)
croisements = sum(transects.geometry.iloc[i].intersects(transects.geometry.iloc[i + 1])
                  for i in range(len(transects) - 1)
                  if transects.troncon.iloc[i] == transects.troncon.iloc[i + 1])
print(f"{len(transects)} transects, {croisements} croisement(s) entre transects voisins")

## 3. Intersections date par date

Pour chaque date, on calcule où le trait de côte coupe chaque transect. La position est exprimée **par rapport au trait de référence de 2024** : positive côté mer, négative côté terre.

In [ ]:
def intersections(transects, traits):
    arbre = STRtree(transects.geometry.values)
    lignes = []
    for i, geom in enumerate(traits.geometry):
        for k in arbre.query(geom):
            inter = transects.geometry.iloc[k].intersection(geom)
            if inter.is_empty:
                continue
            for p in (getattr(inter, "geoms", None) or [inter]):
                if p.geom_type == "Point":
                    lignes.append((i, int(transects.id_transect.iloc[k]),
                                   transects.geometry.iloc[k].project(p) - COTE_TERRE))
    df = pd.DataFrame(lignes, columns=["i_trait", "id_transect", "position"])
    df["date"] = traits.date.values[df.i_trait]
    df["annee"] = traits.annee.values[df.i_trait]
    df["satname"] = traits.satname.values[df.i_trait]
    return df

inter = intersections(transects, traits)
print(f"{len(inter)} intersections")

## 4. Position annuelle robuste

Pour chaque transect et chaque année :
1. médiane de toutes les intersections de l'année ;
2. pour chaque date, on ne garde que l'intersection la plus proche de cette médiane (écarte l'écume, les vagues et les petits fragments) ;
3. on retire les valeurs aberrantes, à plus de 3 écarts absolus médians, puis on recalcule la médiane.

In [ ]:
def positions_annuelles(inter, seuil_mad=3.0):
    m1 = inter.groupby(["id_transect", "annee"]).position.median().rename("m1")
    df = inter.join(m1, on=["id_transect", "annee"])
    df["ecart"] = (df.position - df.m1).abs()
    par_date = df.loc[df.groupby(["id_transect", "i_trait"]).ecart.idxmin()]

    def robuste(s):
        med = s.median()
        mad = (s - med).abs().median()
        if mad > 0:
            s = s[(s - med).abs() <= seuil_mad * 1.4826 * mad]
        return pd.Series({"position": s.median(), "n_dates": len(s),
                          "ecart_type": s.std(ddof=0) if len(s) > 1 else np.nan})

    res = par_date.groupby(["id_transect", "annee"]).position.apply(robuste).unstack().reset_index()
    res["n_dates"] = res.n_dates.astype(int)
    return res

positions = positions_annuelles(inter)
positions = positions.merge(transects[["id_transect", "troncon", "distance_le_long"]], on="id_transect")
print(f"{len(positions)} positions annuelles sur {positions.id_transect.nunique()} transects")

## 5. Un trait de côte par année

Les positions des transects voisins sont reliées pour former le trait de l'année. On coupe la ligne là où un transect manque, ou là où deux positions voisines diffèrent de plus de 60 m.

**Fiabilité :** une année est jugée fiable si elle couvre au moins la moitié des transects. Les années très partielles (une seule image, souvent en partie nuageuse) sont conservées dans le fichier complet, mais exclues de la version pour DSAS.

In [ ]:
def lignes_annuelles(positions, transects, saut_max=60):
    geom = transects.set_index("id_transect").geometry
    lignes = []
    for annee, grp in positions.groupby("annee"):
        grp = grp.sort_values("id_transect")
        pts = np.array([geom.loc[r.id_transect].interpolate(r.position + COTE_TERRE).coords[0]
                        for r in grp.itertuples()])
        ids, tronc = grp.id_transect.values, grp.troncon.values
        coupes = np.where((np.diff(ids) > 1) | (np.diff(tronc) != 0) |
                          (np.hypot(*np.diff(pts, axis=0).T) > ESPACEMENT + saut_max))[0] + 1
        morceaux = [m for m in np.split(pts, coupes) if len(m) >= 3]
        if morceaux:
            lignes.append({"annee": int(annee),
                           "n_transects": len(grp),
                           "couverture": round(len(grp) / len(transects), 2),
                           "n_dates_median": float(grp.n_dates.median()),
                           "geometry": MultiLineString([LineString(m) for m in morceaux])})
    gdf = gpd.GeoDataFrame(lignes, crs=EPSG)
    gdf["fiable"] = gdf.couverture >= 0.5
    return gdf

annuels = lignes_annuelles(positions, transects)
print(annuels.drop(columns="geometry").to_string(index=False))
print(f"\n{annuels.fiable.sum()} années fiables sur {len(annuels)}")

## 6. Exports

- **Fichier complet** : toutes les années, avec leurs indicateurs de fiabilité.
- **Version DSAS** : années fiables, avec une date au format `MM/JJ/AAAA` (1er juillet, milieu d'année) et une incertitude de position (15 m pour les années Landsat, jusqu'en 2015, et 10 m ensuite avec Sentinel-2).
- **Ligne de base DSAS** : la ligne qui relie l'origine des transects, côté terre.
- **Version web** : années fiables en WGS 84, simplifiées à 3 m, pour l'application.

In [ ]:
dossier = os.path.join(ROOT, "data", "processed")
os.makedirs(os.path.join(ROOT, "app", "data"), exist_ok=True)
os.makedirs(os.path.join(ROOT, "outputs", "figures"), exist_ok=True)

# Fichier complet, transects et positions
annuels.to_file(os.path.join(dossier, "traits_de_cote_annuels_1984_2026.geojson"), driver="GeoJSON")
transects.to_file(os.path.join(dossier, "transects.geojson"), driver="GeoJSON")
positions.round(1).to_csv(os.path.join(dossier, "positions_annuelles.csv"), index=False)

# Version DSAS
dsas = annuels[annuels.fiable].copy()
dsas["date_dsas"] = dsas.annee.map(lambda a: f"07/01/{a}")
dsas["incertitude"] = np.where(dsas.annee <= 2015, 15.0, 10.0)
dsas[["annee", "date_dsas", "incertitude", "n_transects", "n_dates_median", "geometry"]].to_file(
    os.path.join(dossier, "traits_de_cote_annuels_dsas.geojson"), driver="GeoJSON")

# Ligne de base DSAS : origines des transects, tronçon par tronçon
base = []
for t, grp in transects.groupby("troncon"):
    if len(grp) >= 2:
        base.append({"troncon": int(t), "geometry": LineString([g.coords[0] for g in grp.geometry])})
gpd.GeoDataFrame(base, crs=EPSG).to_file(os.path.join(dossier, "ligne_de_base_dsas.geojson"), driver="GeoJSON")

# Version web, en WGS 84
web = annuels[annuels.fiable][["annee", "n_dates_median", "geometry"]].copy()
web["geometry"] = web.geometry.simplify(3)
fn_web = os.path.join(ROOT, "app", "data", "traits_de_cote_annuels.geojson")
web.to_crs(4326).to_file(fn_web, driver="GeoJSON", COORDINATE_PRECISION=6)

print(f"{len(dsas)} années exportées pour DSAS")
print(f"Version web : {os.path.getsize(fn_web) / 1e3:.0f} Ko")

## 7. Carte des traits de côte annuels

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 12))
norm = plt.Normalize(annuels.annee.min(), annuels.annee.max())
fiables = annuels[annuels.fiable]
for ax, zoom in zip(axes, [None, "sud"]):
    for r in fiables.itertuples():
        for l in r.geometry.geoms:
            c = np.asarray(l.coords)
            ax.plot(c[:, 0], c[:, 1], color=plt.cm.viridis(norm(r.annee)), lw=0.9)
    ax.set_aspect("equal")
    ax.set_axis_off()
    if zoom:
        y0 = fiables.total_bounds[1]
        ax.set_ylim(y0, y0 + 5000)
        ax.set_xlim(fiables.total_bounds[0] - 100, fiables.total_bounds[0] + 1300)
        ax.set_title("Sud de la Langue de Barbarie (5 km)")
    else:
        ax.set_title("Traits de côte annuels, 1984-2026")
sm = plt.cm.ScalarMappable(cmap="viridis", norm=norm)
fig.colorbar(sm, ax=axes, shrink=0.5, label="Année")
fig.savefig(os.path.join(ROOT, "outputs", "figures", "traits_annuels_1984_2026.png"), dpi=200, bbox_inches="tight")
plt.show()

## 8. Évolution le long de la côte

À gauche : déplacement net de la côte entre le début (1984-1990) et la fin (2022-2026) de la série, en médiane sur chaque période, du sud au nord. À droite : l'évolution année par année sur trois transects (recul maximal, avancée maximale, secteur stable au nord).

Le bilan porte sur la Langue de Barbarie, au nord de l'embouchure. Au sud de l'embouchure, la comparaison n'a pas de sens : dans les années 1980, la flèche se prolongeait plus au sud et la rive actuelle n'était pas une côte océanique. Ce secteur sera étudié avec le suivi de la brèche.

Ce n'est qu'un aperçu : les taux d'évolution détaillés (NSM, EPR, LRR) seront calculés avec DSAS dans ArcGIS Pro.

In [ ]:
# Tronçon principal : la Langue de Barbarie, au nord de l'embouchure (le plus long)
principal = transects.troncon.value_counts().idxmax()
fiab = positions[positions.annee.isin(annuels[annuels.fiable].annee) & (positions.troncon == principal)]
debut = fiab[fiab.annee <= 1990].groupby("id_transect").position.median()
fin = fiab[fiab.annee >= 2022].groupby("id_transect").position.median()
bilan = (fin - debut).rename("deplacement").to_frame().join(transects.set_index("id_transect"))
nord = transects.geometry.apply(lambda g: g.interpolate(COTE_TERRE).y)
bilan["y_km"] = nord.loc[bilan.index].values / 1000

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 7), gridspec_kw={"width_ratios": [1, 1.6]})
couleurs = np.where(bilan.deplacement < 0, "tab:red", "tab:blue")
ax1.barh(bilan.y_km, bilan.deplacement, height=0.045, color=couleurs)
ax1.axvline(0, color="k", lw=0.6)
ax1.set_xlabel("Déplacement net 1984-1990 → 2022-2026 (m)\nrouge : recul, bleu : avancée vers la mer")
ax1.set_ylabel("Nord (km, UTM 28N)")
ax1.grid(alpha=0.3)

bilan = bilan.dropna(subset=["deplacement"])
stable_nord = bilan[bilan.y_km > bilan.y_km.quantile(0.8)].deplacement.abs().idxmin()
exemples = {"recul maximal": bilan.deplacement.idxmin(), "avancée maximale": bilan.deplacement.idxmax(),
            "secteur stable": stable_nord}
for nom, k in exemples.items():
    s = fiab[fiab.id_transect == k].sort_values("annee")
    ax2.plot(s.annee, s.position, "o-", ms=3, label=f"{nom} : transect {k} (nord {bilan.y_km.loc[k]:.1f} km)")
ax2.axhline(0, color="k", lw=0.6)
ax2.set_ylabel("Position par rapport au trait 2024 (m, + = vers la mer)")
ax2.legend(loc="upper center", bbox_to_anchor=(0.5, -0.08), ncol=1, frameon=False)
ax2.grid(alpha=0.3)
fig.savefig(os.path.join(ROOT, "outputs", "figures", "evolution_trait_de_cote.png"), dpi=200, bbox_inches="tight")
plt.show()

print(f"Recul maximal : {-bilan.deplacement.min():.0f} m | avancée maximale : {bilan.deplacement.max():.0f} m")
print(f"Transects en recul : {(bilan.deplacement < -20).mean():.0%} | stables (±20 m) : {(bilan.deplacement.abs() <= 20).mean():.0%} | en avancée : {(bilan.deplacement > 20).mean():.0%}")

## Prochaine étape

Dans ArcGIS Pro, avec DSAS :
1. importer `ligne_de_base_dsas.geojson` et `traits_de_cote_annuels_dsas.geojson` dans une géodatabase fichier ;
2. utiliser `date_dsas` comme champ de date et `incertitude` comme champ d'incertitude ;
3. générer les transects et calculer NSM, EPR et LRR.

La version web (`app/data/traits_de_cote_annuels.geojson`) alimentera le curseur temporel de l'application.